In [5]:
from bs4 import BeautifulSoup
import csv

html_text = '''
<div class="property">
    <div class="property-content-title"><h3>精装三房拎包入住</h3></div>
    <div class="property-content-info-comm-name">万科金域学府</div>
    <div class="property-content-info-comm"><p>南岸学府大道</p></div>
    <div class="property-content-detail">
        <p class="property-content-info-text property-content-info-attribute">3室2厅</p>
        <div>
            <div><p>96㎡</p></div>
            <div><p>高楼层/32层</p></div>
            <div><p>2019年建造</p></div>
        </div>
    </div>
    <div class="property-price-total">115万</div>
</div>
<div class="property">
    <div class="property-content-title"><h3>两室小户型</h3></div>
    <div class="property-content-info-comm-name">融创文旅城</div>
    <div class="property-content-info-comm"><p>沙坪坝西永</p></div>
    <div class="property-content-detail">
        <p class="property-content-info-text property-content-info-attribute">2室1厅</p>
        <div>
            <div><p>72㎡</p></div>
            <div><p>中楼层/18层</p></div>
            <div><p></p></div>
        </div>
    </div>
    <div class="property-price-total">78万</div>
</div>
'''

soup = BeautifulSoup(html_text, "html.parser")
item_list = soup.select(".property")

data_list = []
for item in item_list:
    title = item.select_one(".property-content-title h3").get_text(strip=True)
    house = item.select_one(".property-content-info-comm-name").get_text(strip=True)
    address = item.select_one(".property-content-info-comm p").get_text(strip=True)
    struct = item.select_one(".property-content-detail p.property-content-info-text.property-content-info-attribute").get_text(strip=True)
    area = item.select_one(".property-content-detail > div > div:nth-child(1) > p").get_text(strip=True)
    floor = item.select_one(".property-content-detail > div > div:nth-child(2) > p").get_text(strip=True)
    
    maketime_tag = item.select_one(".property-content-detail > div > div:nth-child(3) > p")
    maketime = maketime_tag.get_text(strip=True) if maketime_tag and maketime_tag.get_text(strip=True) else "无"
    
    price = item.select_one(".property-price-total").get_text(strip=True)

    row = [title, house, address, struct, area, floor, maketime, price]
    data_list.append(row)
    print(row)

with open("house.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["标题","小区","地址","户型","面积","楼层","建造年代","价格"])
    writer.writerows(data_list)
print("✅ 解析完成，house.csv 已生成")


['精装三房拎包入住', '万科金域学府', '南岸学府大道', '3室2厅', '96㎡', '高楼层/32层', '2019年建造', '115万']
['两室小户型', '融创文旅城', '沙坪坝西永', '2室1厅', '72㎡', '中楼层/18层', '无', '78万']
✅ 解析完成，house.csv 已生成


In [4]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin
import csv

base_url = "http://www.bspider.top/anjuke/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Referer": "http://www.bspider.top/anjuke/",
}

current_url = base_url
page_count = 1
all_house = []   # 用来存全部房源

while True:
    print(f"\n=====正在采集第{page_count}页，current_url={current_url}=====")
    try:
        r = requests.get(current_url, headers=headers, timeout=10)
        r.raise_for_status()
    except requests.exceptions.Timeout:
        print("❌ 请求超时，网络无法连通网页，停止采集")
        break
    except Exception as e:
        print(f"❌ 请求异常：{e}")
        break
        
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")

    items = soup.select("div.property")
    if not items:
        print("当前页面无房源，停止采集")
        break

    for first in items:
        title       = first.select_one("h3.property-content-title-name").get_text(strip=True)
        estate      = first.select_one("p.property-content-info-comm-name").get_text(strip=True)
        total_price = first.select_one("p.property-price-total").get_text(strip=True)
        info_ps = first.select("div.property-content-info p.property-content-info-text")
        layout      = info_ps[0].get_text(strip=True)
        area        = info_ps[1].get_text(strip=True)
        orientation = info_ps[2].get_text(strip=True)
        floor       = info_ps[3].get_text(strip=True)
        unit_price  = first.select_one("p.property-price-average").get_text(strip=True)
        year = None
        for p in info_ps:
            m = re.search(r"(\d{4})年建造", p.get_text(strip=True))
            if m:
                year = m.group(1)
                break
        house = {"标题": title, "户型": layout, "面积": area, "朝向": orientation,
                 "楼层": floor, "建造年代": year, "总价": total_price, "单价": unit_price}
        print(house)
        all_house.append(house)  # 存入列表

    next_a = soup.select_one("a.next-active")
    if not next_a:
        print("✅未找到下一页a标签，到达末页，采集终止")
        break
    href_raw = next_a["href"]
    print(f"下一页href原值：{href_raw}")

    next_full_url = urljoin(base_url, href_raw)
    print(f"urljoin补全后的完整地址：{next_full_url}")

    current_url = next_full_url
    page_count += 1

print("\n====全部页面采集完成====")
print("✅一共采集房源数量：", len(all_house))   


with open("house.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["标题","户型","面积","朝向","楼层","建造年代","总价","单价"])
    writer.writeheader()
    writer.writerows(all_house)
print("✅数据已保存到 house.csv")



=====正在采集第1页，current_url=http://www.bspider.top/anjuke/=====
{'标题': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '户型': '3室2厅1卫', '面积': '89.9㎡', '朝向': '南北', '楼层': '低层(共27层)', '建造年代': '2019', '总价': '148万', '单价': '16465元/㎡'}
{'标题': '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '户型': '3室2厅2卫', '面积': '94.5㎡', '朝向': '东南', '楼层': '低层(共28层)', '建造年代': '2008', '总价': '140万', '单价': '14815元/㎡'}
{'标题': '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '户型': '3室2厅1卫', '面积': '88㎡', '朝向': '南北', '楼层': '低层(共24层)', '建造年代': '2016', '总价': '145万', '单价': '16487元/㎡'}
{'标题': '中交中央公园 横厅精装大两房 出门都是公园', '户型': '2室2厅1卫', '面积': '91㎡', '朝向': '南', '楼层': '低层(共25层)', '建造年代': '2016', '总价': '188万', '单价': '20660元/㎡'}
{'标题': '富力现代广场 豪装大两房 户型方正 看房方便', '户型': '2室2厅1卫', '面积': '85.5㎡', '朝向': '南北', '楼层': '低层(共30层)', '建造年代': '2012', '总价': '145万', '单价': '16969元/㎡'}
{'标题': '简奢风格，富力现代广场，正规大2房，住家安静，带，随时看', '户型': '2室2厅1卫', '面积': '85.5㎡', '朝向': '南北', '楼层': '低层(共33层)', '建造年代': '2009', '总价': '145万', '单价': '16969元/㎡'}
{'标题': '6219 鲁能星城一街区 住家装修两房 随时看房', '户型': '2室2厅1卫', '面积': '83.6㎡', 

In [7]:
html_fragment = '''
<div class="property-content-detail">
    <p class="property-content-info-text property-content-info-attribute">3室2厅</p>
    <div>
        <div><p>96㎡</p></div>
        <div><p>高楼层/32层</p></div>
        <div><p>2019年建造</p></div>
    </div>
</div>
'''
print(html_fragment)



<div class="property-content-detail">
    <p class="property-content-info-text property-content-info-attribute">3室2厅</p>
    <div>
        <div><p>96㎡</p></div>
        <div><p>高楼层/32层</p></div>
        <div><p>2019年建造</p></div>
    </div>
</div>



In [8]:
html_price = '''
<div class="property-price-total">115万</div>
'''
print(html_price)



<div class="property-price-total">115万</div>



In [12]:
import csv
with open("house.csv","r",encoding="utf-8-sig") as f:
    r = csv.DictReader(f)
    print(r.fieldnames)


['标题', '小区', '地址', '户型', '面积', '楼层', '建造年代', '价格']


In [13]:
import csv

data_list = []
with open("house.csv","r",encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        item = (
            row["标题"],
            row["小区"],
            row["地址"],
            row["户型"],
            row["面积"],
            row["楼层"],
            row["建造年代"],
            row["价格"]
        )
        data_list.append(item)
print("待入库条数：", len(data_list))


待入库条数： 2


In [16]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin
import csv

base_url = "http://www.bspider.top/anjuke/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Referer": "http://www.bspider.top/anjuke/",
}

current_url = base_url
page_count = 1
all_house = []

while True:
    print(f"\n=====正在采集第{page_count}页，current_url={current_url}=====")
    try:
        r = requests.get(current_url, headers=headers, timeout=10)
        r.raise_for_status()
    except requests.exceptions.Timeout:
        print("❌ 请求超时，网络无法连通网页，停止采集")
        break
    except Exception as e:
        print(f"❌ 请求异常：{e}")
        break
        
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")

    items = soup.select("div.property")
    if not items:
        print("当前页面无房源，停止采集")
        break

    for first in items:
        title       = first.select_one("h3.property-content-title-name").get_text(strip=True)
        estate      = first.select_one("p.property-content-info-comm-name").get_text(strip=True)
        total_price = first.select_one("p.property-price-total").get_text(strip=True)
        info_ps = first.select("div.property-content-info p.property-content-info-text")
        layout      = info_ps[0].get_text(strip=True)
        area        = info_ps[1].get_text(strip=True)
        orientation = info_ps[2].get_text(strip=True)
        floor       = info_ps[3].get_text(strip=True)
        unit_price  = first.select_one("p.property-price-average").get_text(strip=True)
        year = None
        for p in info_ps:
            m = re.search(r"(\d{4})年建造", p.get_text(strip=True))
            if m:
                year = m.group(1)
                break
        house = {"标题": title, "户型": layout, "面积": area, "朝向": orientation,
                 "楼层": floor, "建造年代": year, "总价": total_price, "单价": unit_price}
        all_house.append(house)

    next_a = soup.select_one("a.next-active")
    if not next_a:
        print("✅未找到下一页a标签，到达末页，采集终止")
        break
    href_raw = next_a["href"]
    next_full_url = urljoin(base_url, href_raw)
    current_url = next_full_url
    page_count += 1

print("\n====全部页面采集完成====")
print("✅一共采集房源数量：", len(all_house))   

with open("house.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["标题","户型","面积","朝向","楼层","建造年代","总价","单价"])
    writer.writeheader()
    writer.writerows(all_house)
print("✅数据已保存到 house.csv")





=====正在采集第1页，current_url=http://www.bspider.top/anjuke/=====

=====正在采集第2页，current_url=http://www.bspider.top/anjuke/?page=2=====

=====正在采集第3页，current_url=http://www.bspider.top/anjuke/?page=3=====

=====正在采集第4页，current_url=http://www.bspider.top/anjuke/?page=4=====

=====正在采集第5页，current_url=http://www.bspider.top/anjuke/?page=5=====

=====正在采集第6页，current_url=http://www.bspider.top/anjuke/?page=6=====

=====正在采集第7页，current_url=http://www.bspider.top/anjuke/?page=7=====

=====正在采集第8页，current_url=http://www.bspider.top/anjuke/?page=8=====

=====正在采集第9页，current_url=http://www.bspider.top/anjuke/?page=9=====

=====正在采集第10页，current_url=http://www.bspider.top/anjuke/?page=10=====
✅未找到下一页a标签，到达末页，采集终止

====全部页面采集完成====
✅一共采集房源数量： 600
✅数据已保存到 house.csv


In [18]:
!pip install pyodbc


Defaulting to user installation because normal site-packages is not writeable


In [3]:
%pip install pymssql


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 957.2 kB/s  0:00:03 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pymysql
conn = pymysql.connect(
    host="mysql",
    port=3306,
    user="student",
    password="123456",
    database="smartpark_lab",
    charset="utf8mb4"
)
cur = conn.cursor()

# 删除旧的houseinfo表
cur.execute("DROP TABLE IF EXISTS houseinfo;")

# 新建表，每个中文字段左右都是键盘左上角【`】反引号，不是单引号
sql_create = '''
CREATE TABLE houseinfo (
    `标题` VARCHAR(255),
    `户型` VARCHAR(50),
    `面积` VARCHAR(50),
    `朝向` VARCHAR(50),
    `楼层` VARCHAR(100),
    `建造年代` VARCHAR(30),
    `总价` VARCHAR(50),
    `单价` VARCHAR(50),
    PRIMARY KEY (`标题`,`户型`,`面积`,`朝向`,`楼层`,`建造年代`,`总价`,`单价`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
'''
cur.execute(sql_create)
conn.commit()
cur.close()
conn.close()
print("✅houseinfo 表重建好了")

✅houseinfo 表重建好了


In [5]:
import csv

csv_path = "house.csv"
house_data_list = []

with open(csv_path, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        record = (
            row["标题"],
            row["户型"],
            row["面积"],
            row["朝向"],
            row["楼层"],
            row["建造年代"],
            row["总价"],
            row["单价"]
        )
        house_data_list.append(record)

print(f"一共读取 {len(house_data_list)} 条房源")

一共读取 600 条房源


In [6]:
import pymysql

with pymysql.connect(
    host="mysql", 
    port=3306,
    user="student",
    password="123456",
    database="smartpark_lab",
    charset="utf8mb4", 
    connect_timeout=5
) as conn:
    with conn.cursor() as cursor:
        cursor.execute("SELECT VERSION(), DATABASE(), CURRENT_USER()")
        print(cursor.fetchone())

('8.4.11', 'smartpark_lab', 'student@%')


In [7]:
import pymysql
conn = pymysql.connect(
    host="mysql",
    port=3306,
    user="student",
    password="123456",
    database="smartpark_lab",
    charset="utf8mb4"
)
cur = conn.cursor()

sql_create = '''
CREATE TABLE IF NOT EXISTS houseinfo (
    `标题` VARCHAR(255),
    `户型` VARCHAR(50),
    `面积` VARCHAR(50),
    `朝向` VARCHAR(50),
    `楼层` VARCHAR(100),
    `建造年代` VARCHAR(30),
    `总价` VARCHAR(50),
    `单价` VARCHAR(50),
    PRIMARY KEY (`标题`,`户型`,`面积`,`朝向`,`楼层`,`建造年代`,`总价`,`单价`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
'''
cur.execute(sql_create)
conn.commit()
cur.close()
conn.close()
print("✅houseinfo 表创建完成")

✅houseinfo 表创建完成


In [8]:
import pymysql
conn = pymysql.connect(
    host="mysql",
    port=3306,
    user="student",
    password="123456",
    database="smartpark_lab",
    charset="utf8mb4"
)
cur = conn.cursor()

sql_insert = """
INSERT IGNORE INTO houseinfo
(`标题`,`户型`,`面积`,`朝向`,`楼层`,`建造年代`,`总价`,`单价`)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
"""
try:
    cur.executemany(sql_insert, house_data_list)
    conn.commit()
    print(f"批量入库完成，处理记录数：{cur.rowcount}")
except Exception as e:
    conn.rollback()
    print("入库失败，执行回滚：", e)
finally:
    cur.close()
    conn.close()

批量入库完成，处理记录数：500
